# MID training — CVAE decoder ablation (LOO CV)

Train the **CVAE decoder ablation** of MID: same Trajectron++ encoder as the diffusion model, but the diffusion decoder is swapped out for a small CVAE. This is the paper's **Table 3 group-4** ablation.

Structurally identical to `train.ipynb`, with two differences:
1. `CVAEAutoEncoder` instead of `AutoEncoder` (decoder swap).
2. **Leave-one-out cross-validation** instead of single-scene training. `iter_loo_folds` builds the merged train env (4 scenes) and the held-out test env on each iteration. By default we run just one fold (`SINGLE_FOLD="eth"`); set it to `None` to run all 5.

Smaller batch size than the diffusion run (paper hint — the CVAE is less data-hungry and trains stably at smaller batches).

## 1. Configuration

In [4]:
SINGLE_FOLD = "eth"        # set to None to run all 5 LOO folds
BATCH_SIZE = 64            # smaller than 256 (diffusion) — paper hint
EPOCHS = 10                # smoke-test value
LR = 1e-3
ENCODER_DIM = 256
Z_DIM = 32
KL_WEIGHT = 1.0
AUGMENT = False
NUM_SAMPLES = 20           # for eval Best-of-K
SEED = 123

CHECKPOINT_DIR = "../checkpoints"
PROCESSED_DATA = "../processed_data"

## 2. Setup: paths, device, seeds

`PROJECT_ROOT` (this notebook's parent directory) needs to be on `sys.path` so `import mid_model`, `import environment`, `import models`, `import dataset`, and `import utils` all resolve to the local copies at the project root. The MID repo at `MID/` is no longer needed at runtime — everything we use has been pulled out.

In [5]:
import os
import sys
import time
import random
import numpy as np
import torch

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Pick the best device available.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

Device: mps


## 3. LOO cross-validation loop

Each fold is independent: a fresh `ModelRegistrar`, a fresh `Trajectron` encoder, a fresh `CVAEAutoEncoder`, and a fresh optimizer. The encoder must be re-built per fold because `set_environment(...)` registers one MGCVAE sub-model per node type against the *training* environment of that fold — we can't reuse the encoder across folds with different scene compositions.

Evaluation runs on the held-out scene's test split. We pass `eth_rescale=True` only when the held-out scene is `eth` (the ETH test set was scaled by 0.6 during preprocessing; the literature reports numbers in the unscaled frame).

In [3]:
from tqdm.auto import tqdm

from mid_model import (
    build_dataloader,
    get_hyperparameters,
    CVAEAutoEncoder,
    evaluate,
    iter_loo_folds,
)
from models.trajectron import Trajectron
from utils.model_registrar import ModelRegistrar

fold_results = []  # collected (held_out, ade, fde) tuples

for fold_idx, (held_out, train_env, test_env) in enumerate(
    iter_loo_folds(PROCESSED_DATA, single_fold=SINGLE_FOLD)
):
    print(f"\n{'='*60}")
    print(f"  Fold {fold_idx + 1}: held-out scene = {held_out!r}")
    print(f"  train scenes: {len(train_env.scenes)}  |  test scenes: {len(test_env.scenes)}")
    print(f"{'='*60}")

    # ---- hyperparams + dataloaders ----
    hyperparams = get_hyperparameters(encoder_dim=ENCODER_DIM)
    hyperparams["batch_size"] = BATCH_SIZE

    train_loader, node_type = build_dataloader(
        env=train_env,
        hyperparams=hyperparams,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        augment=AUGMENT,
    )
    test_loader, _ = build_dataloader(
        env=test_env,
        hyperparams=hyperparams,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        augment=False,
    )
    print(f"  batches/epoch: {len(train_loader)}  |  test batches: {len(test_loader)}")

    # ---- fresh encoder for this fold ----
    registrar = ModelRegistrar(model_dir=CHECKPOINT_DIR, device=DEVICE)
    encoder = Trajectron(registrar, hyperparams, DEVICE)
    encoder.set_environment(train_env)
    encoder.set_annealing_params()

    # ---- assemble CVAE model ----
    model = CVAEAutoEncoder(
        encoder=encoder,
        registrar=registrar,
        encoder_dim=ENCODER_DIM,
        z_dim=Z_DIM,
        kl_weight=KL_WEIGHT,
    ).to(DEVICE)
    print(f"  params: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    # ---- smoke test ----
    batch = next(iter(train_loader))
    model.train()
    loss = model.get_loss(batch, node_type)
    assert torch.isfinite(loss), "smoke-test loss is not finite"
    print(f"  smoke-test loss: {loss.item():.4f}")

    # ---- training loop ----
    history = []
    for epoch in range(1, EPOCHS + 1):
        epoch_losses = []
        t0 = time.time()
        pbar = tqdm(train_loader, desc=f"  [{held_out}] epoch {epoch}/{EPOCHS}", ncols=100)
        for batch in pbar:
            optimizer.zero_grad()
            loss = model.get_loss(batch, node_type)
            loss.backward()
            optimizer.step()
            epoch_losses.append(loss.item())
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = float(np.mean(epoch_losses))
        elapsed = time.time() - t0
        history.append(avg_loss)
        print(f"    epoch {epoch}: avg loss = {avg_loss:.4f}  ({elapsed:.1f}s)")

    # ---- evaluate on held-out scene ----
    eth_rescale = (held_out == "eth")
    results = evaluate(
        model=model,
        dataloader=test_loader,
        node_type=node_type,
        device=DEVICE,
        sample=NUM_SAMPLES,
        eth_rescale=eth_rescale,
    )
    ade = float(results["ade"])
    fde = float(results["fde"])
    print(f"  [{held_out}] Best-of-{NUM_SAMPLES}  ADE={ade:.4f}  FDE={fde:.4f}")
    fold_results.append((held_out, ade, fde))

    # ---- save checkpoint ----
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"cvae_loo_{held_out}.pt")
    torch.save({
        "scene": held_out,
        "epoch": EPOCHS,
        "hyperparams": hyperparams,
        "encoder_dim": ENCODER_DIM,
        "z_dim": Z_DIM,
        "kl_weight": KL_WEIGHT,
        "registrar_state_dict": registrar.model_dict.state_dict(),
        "cvae_state_dict": model.cvae.state_dict(),
        "history": history,
        "ade": ade,
        "fde": fde,
    }, ckpt_path)
    print(f"  saved: {ckpt_path}")

/opt/anaconda3/envs/ml_hw/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



  Fold 1: held-out scene = 'eth'
  train scenes: 27  |  test scenes: 1
  batches/epoch: 1873  |  test batches: 20
  params: 835,180
  smoke-test loss: 0.8791


  [eth] epoch 1/10:   6%|█▋                         | 117/1873 [00:23<05:58,  4.90it/s, loss=0.0942]


KeyboardInterrupt: 

## 4. Summary

Per-fold ADE/FDE. If multiple folds ran, also report the mean across folds.

In [ ]:
print("────────────────────────────────────────")
print("    Fold    ADE      FDE")
print("────────────────────────────────────────")
for held_out, ade, fde in fold_results:
    print(f"    {held_out:<7} {ade:<8.4f} {fde:<8.4f}")
if len(fold_results) > 1:
    mean_ade = float(np.mean([r[1] for r in fold_results]))
    mean_fde = float(np.mean([r[2] for r in fold_results]))
    print("────────────────────────────────────────")
    print(f"    {'mean':<7} {mean_ade:<8.4f} {mean_fde:<8.4f}")
print("────────────────────────────────────────")